# Complaint Triage System: training di Google Colab

Jalankan dengan GPU gratis: **Runtime → Change runtime type → T4 GPU**.

Notebook ini hanya memanggil skrip di repo, jadi hasilnya sama dengan menjalankan skrip di laptop.
Data tidak ada di GitHub: unggah `brimo.jsonl` hasil `import_reviews.py` + anotasi urgensi di sel ke-3.

In [ ]:
!git clone https://github.com/HachiVin/Complaint-Triage-System.git
%cd Complaint-Triage-System
!pip install -q -r requirements-train.txt
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada")

In [ ]:
# Unggah data/annotated/brimo.jsonl (dan opsional bank_lain.jsonl untuk uji eksternal)
import os, shutil
from google.colab import files
os.makedirs("data/annotated", exist_ok=True)
for name in files.upload():
    shutil.move(name, f"data/annotated/{name}")
!ls -la data/annotated

In [ ]:
# Time split: 15% ulasan terbaru jadi test set
EXTERNAL = "--external data/annotated/bank_lain.jsonl" if os.path.exists("data/annotated/bank_lain.jsonl") else ""
!python scripts/prepare_data.py data/annotated/brimo.jsonl data/processed {EXTERNAL}
!cat data/processed/split_report.json

In [ ]:
# 1) Baseline TF-IDF (hitungan detik)
!python scripts/train_baseline.py data/processed models/baseline --version v1

In [ ]:
# 2) IndoBERT multi-task: aspek + sentimen + urgensi dalam satu encoder
# 100k ulasan x 4 epoch cukup lama di T4; mulai dengan --max-train-rows 30000 untuk percobaan pertama.
!python scripts/train_multitask.py data/processed models/indobert-mt --epochs 3 --max-train-rows 30000

In [ ]:
# 3) Bandingkan di test set (dan set eksternal) -> docs/results.md
!python scripts/evaluate.py data/processed --baseline models/baseline --multitask models/indobert-mt
print(open("docs/results.md").read())

In [ ]:
# 4) Unduh model dan hasil evaluasi
!zip -qr triage-models.zip models docs/results.md docs/results.json
files.download("triage-models.zip")